# KION AutoFE: прогон на Colab T4
Runtime → Change runtime type → **T4 GPU**. Долгие этапы запускаются в фоне (`nohup`), прогресс — ячейкой статуса. Этапы возобновляемые: база и кандидаты кэшируются в `cache/`, каждый шаг пишет `results/n<N>/<step>.json`.

In [ ]:
!nvidia-smi -L
!test -d /content/kion-autofe || git clone -q https://github.com/r3dmistyle/kion-autofe.git /content/kion-autofe
%cd /content/kion-autofe
!git pull -q
!pip -q install duckdb catboost tabulate
!python -c "import numpy as np; from catboost import CatBoostClassifier as C; X=np.random.rand(2000,5); C(iterations=20, task_type='GPU', verbose=0).fit(X, X[:,0]>.5); print('catboost GPU ok')"
!mkdir -p logs && python tests/test_leakage.py && python src/ranker.py && python src/search.py

In [ ]:
# smoke: whole pipeline on 25k users with a tiny search budget (~10 min)
!python src/experiments.py --n-users 25000 --device gpu --seeds 1 --train-weeks 1 --chunk 1000 --pool 10 --mut-fits 3 2>&1 | grep -vi warning | tail -40
!rm -rf results/n25000

In [ ]:
# main run in the background: baseline, hand-crafted, AutoFE search, comparison, window sweep
!nohup python src/experiments.py --n-users 100000 --device gpu --steps e0,hand,search,auto,windows > logs/n100000.log 2>&1 &

In [ ]:
# status
!grep -vi warning logs/n100000.log | tail -15; ls results/*/ 2>/dev/null; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; free -g | head -2

In [ ]:
# scaling: same pipeline, features selected at 100k reused
!nohup sh -c 'for n in 25000 50000 200000; do python src/experiments.py --n-users $n --device gpu --seeds 1 --steps e0,hand,auto --search-from results/n100000 > logs/n$n.log 2>&1; done' > /dev/null 2>&1 &

In [ ]:
!python src/experiments.py --steps report
!zip -qr results.zip results logs && ls -la results.zip